<div style="
    background: linear-gradient(135deg, #2D1E2F 0%, #5A2A5E 45%, #2C6E8F 100%);
    padding: 34px;
    border-radius: 14px;
    margin: 10px 0 20px 0;
    box-shadow: 0 10px 26px rgba(45,30,47,0.34);
    border: 1px solid rgba(255,255,255,0.12);
">
    <h1 style="
        color: #ffffff;
        margin: 0;
        text-align: center;
        font-size: 34px;
        font-weight: 300;
        letter-spacing: 2px;
    ">PHY-3500 · Notebook 03</h1>
    <p style="
        color: rgba(255,255,255,0.92);
        margin: 10px 0 0 0;
        text-align: center;
        font-size: 18px;
        letter-spacing: 0.5px;
    ">Autoencodeur MLP pour réduction de dimension spectrale</p>
</div>

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.14), rgba(44,110,143,0.12));
    border: 1px solid rgba(90,42,94,0.30);
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<strong>Équipe :</strong> Alex, Justine<br>
<strong>Cours :</strong> PHY-3500 Physique numérique - Université Laval<br>
<strong>Prérequis :</strong> Notebook 01 exécuté (features + PCAAnalyzer disponibles)
</div>

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.14), rgba(45,30,47,0.14));
    border: 1px solid rgba(44,110,143,0.30);
    border-radius: 10px;
    padding: 18px 20px;
    margin: 14px 0;
">
<h3 style="margin: 0 0 10px 0; color: #5A2A5E;">Objectifs</h3>
<ol style="margin: 0; padding-left: 20px; line-height: 1.6;">
  <li>Entraîner un autoencodeur MLP sur les features spectrales LAMOST DR5.</li>
  <li>Explorer l’espace latent 2D et le comparer à PCA et UMAP.</li>
  <li>Comparer les performances de reconstruction avec PCA.</li>
  <li>Tester la continuité de l’espace latent par interpolation.</li>
  <li>Produire une synthèse comparative des méthodes.</li>
</ol>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 12px 14px;
    margin-top: 10px;
">
<strong>Positionnement scientifique :</strong> ce notebook évalue une réduction de dimension non linéaire paramétrique (autoencodeur) et la confronte à deux baselines non supervisées (PCA, UMAP).
</div>

<a id="toc"></a>

## Table des matières

<div style="display: grid; grid-template-columns: repeat(2, minmax(260px, 1fr)); gap: 12px; margin: 14px 0 22px 0;">

  <div style="background: linear-gradient(135deg, #5A2A5E, #2C6E8F); padding: 14px; border-radius: 10px;">
    <a href="#setup" style="color: white; text-decoration: none;"><strong>0. Imports et configuration</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #2C6E8F, #5A2A5E); padding: 14px; border-radius: 10px;">
    <a href="#data" style="color: white; text-decoration: none;"><strong>1. Chargement des données</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #5A2A5E, #3D405B); padding: 14px; border-radius: 10px;">
    <a href="#arch" style="color: white; text-decoration: none;"><strong>2. Architecture et entraînement</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #3D405B, #2C6E8F); padding: 14px; border-radius: 10px;">
    <a href="#latent" style="color: white; text-decoration: none;"><strong>3. Espace latent 2D</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #2C6E8F, #5A2A5E); padding: 14px; border-radius: 10px;">
    <a href="#recon" style="color: white; text-decoration: none;"><strong>4. Qualité de reconstruction</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #5A2A5E, #3D405B); padding: 14px; border-radius: 10px;">
    <a href="#interp" style="color: white; text-decoration: none;"><strong>5. Continuité latente</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #3D405B, #2C6E8F); padding: 14px; border-radius: 10px;">
    <a href="#synthesis" style="color: white; text-decoration: none;"><strong>6. Synthèse comparative</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #2C6E8F, #5A2A5E); padding: 14px; border-radius: 10px;">
    <a href="#save" style="color: white; text-decoration: none;"><strong>7. Sauvegarde des sorties</strong></a>
  </div>
  <div style="background: linear-gradient(135deg, #5A2A5E, #2D1E2F); padding: 14px; border-radius: 10px; grid-column: span 2;">
    <a href="#conclusion" style="color: white; text-decoration: none;"><strong>Conclusion et références</strong></a>
  </div>

</div>

<a id="setup"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">0 · Imports et configuration</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Cette cellule fixe l’environnement d’exécution, les chemins d’artefacts et la graine aléatoire. C’est la base de la reproductibilité expérimentale du notebook.
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# ── Environnement projet ───────────────────────────────────────────────────
sys.path.insert(0, str(Path("../..").resolve() / "src"))

from utils import setup_project_env, latest_file

paths = setup_project_env(verbose=True)

# ── Chemins ────────────────────────────────────────────────────────────────
FEATURES_PATH = latest_file(paths["PROCESSED_DIR"], "features_*.csv")
if FEATURES_PATH is None:
    raise FileNotFoundError(
        "Aucun fichier features_*.csv trouvé dans data/reports/\n"
        "→ Lancer d'abord 00_master_pipeline.ipynb pour générer les features."
    )
FEATURES_PATH  = Path(FEATURES_PATH)
CATALOG_PATH   = Path(paths["CATALOG_DIR"]) / "master_catalog_gaia.csv"
FIGURES_ROOT   = Path(paths["REPORTS_DIR"]) / "figures" / "phy3500"
FIGURES_DIR    = FIGURES_ROOT / FEATURES_PATH.stem
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR     = Path(paths["REPORTS_DIR"]) / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nFeatures  : {FEATURES_PATH}")
print(f"Catalog   : {CATALOG_PATH}")
print(f"Figures   : {FIGURES_DIR}")
print(f"Models    : {MODELS_DIR}")

# ── Imports dimred ─────────────────────────────────────────────────────────
from dimred import DimRedDataLoader, PCAAnalyzer, DimRedVisualizer
from dimred.autoencoder import SpectralAutoencoder

# ── Logging & reproductibilité ─────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\n✓ Environnement prêt.")

<a id="data"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #3D405B 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">1 · Chargement des données</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Le pipeline de chargement est identique aux notebooks 01 et 02 pour garantir une comparaison rigoureuse entre méthodes.
</div>

Les variables d’entrée sont standardisées avant l’apprentissage :

$$
x'_j = \frac{x_j - \mu_j}{\sigma_j}
$$

Cette étape est essentielle car l’autoencodeur optimise une perte MSE sensible aux échelles des features.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #3D405B; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
loader = DimRedDataLoader(
    features_path=FEATURES_PATH,
    catalog_path=CATALOG_PATH,
    random_state=RANDOM_STATE,
)

X, y, meta = loader.load(
    mode="features",
    snr_min=10.0,
    scale=True,          # StandardScaler — obligatoire pour l'AE
    class_balance=False,
)

feature_names = loader.get_feature_names()
input_dim     = X.shape[1]

print(f"\nDonnées chargées :")
print(f"  X         : {X.shape}")
print(f"  y         : {y.shape} | classes = {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"  meta      : {meta.shape}")
print(f"  input_dim : {input_dim}")

In [ ]:
# Chargement du PCAAnalyzer sauvegardé (depuis notebook 01)
# ── Cherche le modèle PCA dans MODELS_DIR ─────────────────────────────────
pca_model_path = MODELS_DIR / "pca_analyzer.joblib"

if pca_model_path.exists():
    pca = PCAAnalyzer.load(str(pca_model_path))
    scores_pca = pca.transform(X)
    print(f"✓ PCAAnalyzer chargé | {pca.sklearn_pca.n_components_} composantes")
    print(f"  scores_pca : {scores_pca.shape}")
else:
    # Réentraîne PCA si modèle absent
    print("⚠  PCAAnalyzer non trouvé → réentraînement...")
    pca = PCAAnalyzer(n_components=50, random_state=RANDOM_STATE)
    scores_pca = pca.fit_transform(X, feature_names=feature_names)
    pca.save(str(pca_model_path))
    print(f"✓ PCAAnalyzer entraîné et sauvegardé")

n_pcs_95 = pca.n_components_for_variance(0.95)
print(f"  95% variance → {n_pcs_95} composantes PCA")

<a id="arch"></a>

<div style="background: linear-gradient(135deg, #3D405B 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(61,64,91,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">2 · Architecture et entraînement de l’autoencodeur</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(90,42,94,0.10), rgba(44,110,143,0.10));
    border-left: 6px solid #5A2A5E;
    border-radius: 10px;
    padding: 14px 16px;
    margin-bottom: 10px;
">
<strong>Architecture choisie :</strong> X(D) -> 256 -> 128 -> 64 -> z(2) -> 64 -> 128 -> 256 -> Xhat(D)<br>
<strong>Activation :</strong> ReLU + BatchNorm + Dropout(0.1)<br>
<strong>Loss :</strong> MSE<br>
<strong>Optimisation :</strong> Adam + ReduceLROnPlateau + early stopping
</div>

Objectif d’apprentissage (forme simplifiée) :

$$
\mathcal{L}(\theta)=\frac{1}{N}\sum_{i=1}^N\|x_i-\hat x_i\|_2^2 + \lambda\|\theta\|_2^2
$$

où :
- le premier terme force la fidélité de reconstruction ;
- le second (weight decay) régularise les paramètres pour limiter le surapprentissage.

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-bottom: 10px;
">
Le choix de z=2 est volontaire : il impose une compression sévère pour comparer directement Autoencodeur, PCA (PC1/PC2) et UMAP sur un même plan 2D.
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Construction du modèle ────────────────────────────────────────────────
ae = SpectralAutoencoder(
    input_dim=input_dim,
    latent_dim=2,               # 2D pour comparaison directe PCA / UMAP
    hidden_dims=[256, 128, 64],
    dropout=0.1,
)

print(ae)
print(f"\nDevice : {ae.device}")

# Compte des paramètres (info pour le rapport)
try:
    import torch
    n_params = sum(p.numel() for p in ae._model.parameters()
                   if p is not None and p.requires_grad)
    # Reconstruit _model pour le comptage avant fit()
except Exception:
    pass

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────
# Vérifie si un modèle sauvegardé existe déjà
ae_model_path = MODELS_DIR / "ae_latent2.pt"

if ae_model_path.exists():
    print("✓ Modèle autoencodeur trouvé — chargement...")
    ae = SpectralAutoencoder.load(str(ae_model_path))
    history = ae.history_
    print(f"  fit_time : {ae.fit_time_:.1f}s")
    print(f"  best val_loss : {min(history['val_loss']):.6f}")
else:
    print("Entraînement de l'autoencodeur (latent_dim=2)...")
    history = ae.fit(
        X,
        epochs=200,
        lr=1e-3,
        batch_size=512,
        val_fraction=0.10,
        weight_decay=1e-5,
        lr_scheduler=True,
        early_stopping_patience=25,
        verbose=True,
    )
    ae.save(str(ae_model_path))
    print(f"\n✓ Modèle sauvegardé : {ae_model_path}")

print(f"\nEpochs effectuées : {len(history['train_loss'])}")
print(f"Best val_loss     : {min(history['val_loss']):.6f}")

In [ ]:
# ── Courbe d'apprentissage ────────────────────────────────────────────────
viz = DimRedVisualizer(figsize=(12, 4), dpi=150, output_dir=FIGURES_DIR)

fig, axes = viz.plot_training_history(
    history,
    save_path=FIGURES_DIR / "ae_training_history.png",
)
plt.show()

<a id="latent"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">3 · Espace latent 2D - visualisation et interprétation physique</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
Les mêmes colorations physiques que dans PCA et UMAP sont utilisées pour comparer directement les géométries d’embedding.
</div>

Un encodeur apprend une application non linéaire :

$$
f_\theta: \mathbb{R}^{208} \rightarrow \mathbb{R}^{2}, \quad z=f_\theta(x)
$$

et le décodeur reconstruit :

$$
g_\phi: \mathbb{R}^{2} \rightarrow \mathbb{R}^{208}, \quad \hat x=g_\phi(z)
$$

L’objectif est de compresser l’information spectrale utile dans deux variables latentes tout en conservant une structure physiquement cohérente.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Encodage ──────────────────────────────────────────────────────────────
Z_ae = ae.encode(X)
print(f"Espace latent : {Z_ae.shape}")
print(f"  axe 1 : [{Z_ae[:,0].min():.2f}, {Z_ae[:,0].max():.2f}]")
print(f"  axe 2 : [{Z_ae[:,1].min():.2f}, {Z_ae[:,1].max():.2f}]")

In [ ]:
# ── Grille de visualisations ──────────────────────────────────────────────
fig, axes = viz.plot_ae_embedding(
    Z_ae, meta, y,
    save_path=FIGURES_DIR / "ae_latent_grid.png",
)
plt.show()

### 3.1 · Espace latent zoomé — population stellaire

La figure précédente a une échelle globale dominée par quelques **points extrêmes** (4 QSOs, certaines galaxies atypiques) que l'autoencodeur entraîné sur des étoiles ne peut pas reconstruire — il les repousse loin de la région stellaire dans l'espace latent. Cela illustre une propriété utile : l'AE agit implicitement comme un **détecteur d'anomalies**.

Pour visualiser la structure interne de la population stellaire, on zoome sur la région principale (percentiles 1–99).

In [ ]:
# ── Espace latent zoomé (population stellaire principale) ───────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Calcul des bornes robustes (percentile 0.5 / 99.5)
x_lo, x_hi = np.percentile(Z_ae[:, 0], [0.5, 99.5])
y_lo, y_hi = np.percentile(Z_ae[:, 1], [0.5, 99.5])
pad_x = (x_hi - x_lo) * 0.10
pad_y = (y_hi - y_lo) * 0.10

CLASS_PALETTE = {
    "STAR": "#4C72B0", "GALAXY": "#DD8452",
    "QSO": "#55A868",  "UNKNOWN": "#8172B3",
}
CLASS_MARKER  = {"STAR": "o", "GALAXY": "s", "QSO": "^", "UNKNOWN": "x"}
PHYS_PARAMS   = [
    ("T_eff (K)",    "teff_gspphot", "plasma"),
    ("[Fe/H]",       "mh_gspphot",   "RdYlBu_r"),
    ("log g",        "logg_gspphot", "viridis"),
]

fig, axes = plt.subplots(1, 1 + len(PHYS_PARAMS), figsize=(22, 5), dpi=150)

# Panneau 1 : coloration par classe
ax = axes[0]
for cls in np.unique(y):
    mask = y == cls
    ax.scatter(
        Z_ae[mask, 0], Z_ae[mask, 1],
        c=CLASS_PALETTE.get(cls, "#888"),
        marker=CLASS_MARKER.get(cls, "o"),
        s=2, alpha=0.45, linewidths=0,
        label=f"{cls} (n={mask.sum()})", rasterized=True,
    )
ax.set_xlim(x_lo - pad_x, x_hi + pad_x)
ax.set_ylim(y_lo - pad_y, y_hi + pad_y)
ax.legend(markerscale=4, fontsize=8, framealpha=0.9)
ax.set_xlabel("Latent axe 1", fontsize=10)
ax.set_ylabel("Latent axe 2", fontsize=10)
ax.set_title("Type spectral", fontsize=11)
ax.grid(True, alpha=0.25)

# Panneaux 2-4 : paramètres physiques
for ax_i, (label, col, cmap) in zip(axes[1:], PHYS_PARAMS):
    if col not in meta.columns:
        ax_i.axis("off")
        continue
    vals  = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    if (~valid).any():
        ax_i.scatter(
            Z_ae[~valid, 0], Z_ae[~valid, 1],
            c="#e0e0e0", s=1, alpha=0.15, rasterized=True, linewidths=0,
        )
    sc = ax_i.scatter(
        Z_ae[valid, 0], Z_ae[valid, 1],
        c=vals[valid], cmap=cmap,
        vmin=np.nanpercentile(vals[valid], 2),
        vmax=np.nanpercentile(vals[valid], 98),
        s=2, alpha=0.55, linewidths=0, rasterized=True,
    )
    plt.colorbar(sc, ax=ax_i, fraction=0.046, pad=0.04, label=label).ax.tick_params(labelsize=8)
    ax_i.set_xlim(x_lo - pad_x, x_hi + pad_x)
    ax_i.set_ylim(y_lo - pad_y, y_hi + pad_y)
    ax_i.set_xlabel("Latent axe 1", fontsize=10)
    ax_i.set_title(label, fontsize=11)
    ax_i.grid(True, alpha=0.25)

fig.suptitle(
    "Autoencodeur — Espace latent zoomé (population stellaire, P0.5–P99.5)\n"
    "LAMOST DR5 × Gaia DR3",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
zoom_path = FIGURES_DIR / "ae_latent_grid_zoomed.png"
fig.savefig(zoom_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Figure zoomée : {zoom_path}")
print(f"  Fenêtre : x=[{x_lo:.2f}, {x_hi:.2f}]  y=[{y_lo:.2f}, {y_hi:.2f}]")
n_outliers = ((Z_ae[:, 0] < x_lo) | (Z_ae[:, 0] > x_hi) |
              (Z_ae[:, 1] < y_lo) | (Z_ae[:, 1] > y_hi)).sum()
print(f"  Outliers exclus de la fenêtre : {n_outliers} / {len(Z_ae)} "
      f"({100 * n_outliers / len(Z_ae):.2f}%)")

### Interprétation — Espace latent et détection d'anomalies

La version zoomée révèle la structure interne de la population stellaire :
- **Axe latent 2** encode principalement la **température** (ρ Spearman = -0.76 avec T_eff) — le gradient chaud/froid est clairement visible.
- **Axe latent 1** encode une combinaison de métallicité et de stade évolutif (ρ ≈ 0.27 avec [Fe/H]), mais de façon moins nette que PC1 de la PCA.

Les axes latents sont **arbitraires en orientation** (signe et ordre), contrairement aux composantes PCA ordonnées par variance décroissante. C'est une limitation fondamentale des autoencodeurs non supervisés.

**Propriété émergente — détection d'anomalies** : les quelques galaxies et QSOs sont repoussés à des coordonnées latentes extrêmes (jusqu'à y≈88 pour les QSOs). L'AE, entraîné quasi-exclusivement sur des étoiles, ne peut pas reconstruire les spectres non-stellaires → il les projette hors de la distribution apprise. Un seuil simple sur la distance à l'origine dans l'espace latent suffirait à les détecter sans supervision.

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.10), rgba(90,42,94,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 8px 0 14px 0;
">
<h4 style="margin: 0 0 8px 0; color: #2C6E8F;">Corrélations espace latent et paramètres physiques (Spearman)</h4>
Comparer ces corrélations avec PC1/PC2 permet d’évaluer ce que l’autoencodeur préserve (ou transforme) de la structure physique stellaire.
</div>

Le coefficient de Spearman mesure la monotonicité entre deux variables de rang :

$$
\rho_s = 1 - \frac{6\sum_i d_i^2}{n(n^2-1)}
$$

où $d_i$ est la différence entre les rangs de deux observations.

### Interprétation
- Une forte $|\rho_s|$ entre un axe latent et T_eff ou BP-RP indique une capture efficace d’un gradient astrophysique majeur.
- Une redistribution de l’information entre axe 1 et axe 2 est normale en non linéaire : les axes latents ne sont pas directement comparables aux composantes PCA.

In [ ]:
# Corrélations espace latent vs paramètres physiques
from scipy.stats import spearmanr

phys_cols = ["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"]
phys_labels = ["T_eff", "log g", "[Fe/H]", "G_BP-G_RP"]

print("Corrélations Spearman : Espace latent AE ↔ Paramètres physiques")
print("-" * 60)
print(f"{'Paramètre':<15} {'Axe latent 1':>14} {'Axe latent 2':>14}")
print("-" * 60)

for col, lbl in zip(phys_cols, phys_labels):
    if col not in meta.columns:
        continue
    vals = meta[col].values.astype(float)
    valid = np.isfinite(vals)
    r1, _ = spearmanr(Z_ae[valid, 0], vals[valid])
    r2, _ = spearmanr(Z_ae[valid, 1], vals[valid])
    print(f"{lbl:<15} {r1:>+14.3f} {r2:>+14.3f}")

print("-" * 60)

# Comparaison directe avec PC1/PC2
print("\nRappel — PC1/PC2 (notebook 01) :")
print(f"{'Paramètre':<15} {'PC1':>14} {'PC2':>14}")
print("-" * 60)
corr_pca = pca.correlations_with_params(meta[phys_cols], scores=scores_pca, n_pcs=2)
for col, lbl in zip(phys_cols, phys_labels):
    if col not in corr_pca.columns:
        continue
    print(f"{lbl:<15} {corr_pca.loc['PC1', col]:>+14.3f} {corr_pca.loc['PC2', col]:>+14.3f}")

<a id="recon"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #3D405B 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">4 · Qualité de reconstruction - autoencodeur vs PCA</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
<strong>4.1 Exemples de reconstruction</strong>
</div>

Pour chaque objet, l’erreur quadratique moyenne est :

$$
\mathrm{MSE}_i = \frac{1}{D}\sum_{j=1}^{D}(x_{ij}-\hat x_{ij})^2
$$

La comparaison AE vs PCA permet de mesurer le gain de non-linéarité à dimension fixée (2D), puis de situer ce gain face à une PCA plus riche (n composantes pour 95% de variance).

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #2C6E8F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Reconstruction de l'ensemble ─────────────────────────────────────────
X_recon = ae.reconstruct(X)

print(f"MSE globale autoencodeur (latent_dim=2) : {ae.reconstruction_mse(X):.5f}")
print(f"MSE globale PCA (n=2)                   : "
      f"{pca.reconstruction_error(X, n_components=2).mean():.5f}")
print(f"MSE globale PCA (n={n_pcs_95})                  : "
      f"{pca.reconstruction_error(X, n_components=n_pcs_95).mean():.5f}")

In [ ]:
# ── Exemples visuels : original vs reconstruit ────────────────────────────
fig, axes = viz.plot_ae_reconstruction(
    X, X_recon, feature_names,
    n_samples=5,
    y=y,
    save_path=FIGURES_DIR / "ae_reconstruction_examples.png",
)
plt.show()

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>4.2 Comparaison MSE autoencodeur vs PCA</strong>
</div>

In [ ]:
# ── Comparaison AE vs PCA ─────────────────────────────────────────────────
comparison_df = ae.compare_with_pca(
    X, pca,
    n_pcs_list=[1, 2, 3, 5, 8, 10, 15, 20, 30, 50],
)
print(comparison_df.to_string(index=False))

fig, ax = viz.plot_ae_vs_pca(
    comparison_df,
    save_path=FIGURES_DIR / "ae_vs_pca_mse.png",
)
plt.show()

### Interprétation — AE@2 ≈ PCA@10 : la valeur de la non-linéarité

C'est le résultat central de ce notebook :

| Méthode | Dimensions | MSE reconstruction |
|---------|-----------|-------------------|
| PCA     | 2         | 0.7317            |
| PCA     | 5         | 0.6290            |
| PCA     | 8         | 0.5722            |
| **AE**  | **2**     | **0.5411**        |
| PCA     | 10        | 0.5430            |
| PCA     | 15        | 0.4803            |

L'autoencodeur à **2 dimensions latentes** reconstruit les spectres aussi bien qu'une **PCA à 10 composantes**. Autrement dit, la non-linéarité permet d'encoder 5× plus d'information dans le même nombre de dimensions.

Cela s'explique physiquement : la séquence stellaire n'est pas un sous-espace euclidien — c'est une **variété courbée** dans l'espace des features. La PCA doit utiliser de nombreux axes linéaires pour approximer cette courbure, là où l'AE la capture directement.

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>4.3 Distribution des erreurs par classe</strong>
</div>

In [ ]:
# ── Résumé par classe ─────────────────────────────────────────────────────
summary = ae.reconstruction_summary(X, y=y)
print("\nErreurs de reconstruction par classe :")
print(summary.to_string(index=False))

# Distribution
mse_per = ae.reconstruction_mse(X, per_sample=True)

fig, ax = viz.plot_ae_error_distribution(
    mse_per, y,
    save_path=FIGURES_DIR / "ae_error_distribution.png",
)
plt.show()

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>4.4 · Distribution des erreurs — version corrigée (axe logarithmique)</strong>
</div>

La figure précédente avait une échelle linéaire dominée par les QSOs (MSE médiane ≈ 30), rendant la distribution des étoiles invisible. On utilise ici un axe x logarithmique pour visualiser toutes les classes simultanément.

In [ ]:
# ── Distribution des erreurs — version log scale ──────────────────────────
import matplotlib.pyplot as plt
import numpy as np

mse_per = ae.reconstruction_mse(X, per_sample=True)

CLASS_COLORS = {
    "STAR": "#4C72B0", "GALAXY": "#DD8452",
    "QSO": "#55A868",  "UNKNOWN": "#8172B3",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

for ax, xscale in zip(axes, ["linear", "log"]):
    for cls in np.unique(y):
        mask = y == cls
        vals = mse_per[mask]
        med  = np.median(vals)
        color = CLASS_COLORS.get(cls, "gray")
        # bins adaptés à l'échelle
        if xscale == "log":
            bins = np.logspace(
                np.log10(max(vals.min(), 1e-4)), np.log10(vals.max() + 1), 60
            )
        else:
            # Limiter à P99 des étoiles pour l'échelle linéaire
            star_p99 = np.percentile(mse_per[y == "STAR"], 99)
            bins = np.linspace(0, star_p99 * 1.5, 60)
        ax.hist(
            vals, bins=bins, alpha=0.55, color=color, density=True,
            label=f"{cls}  médiane={med:.3f}",
        )
        ax.axvline(med, color=color, lw=1.8, ls="--", alpha=0.9)
    ax.set_xlabel("MSE de reconstruction", fontsize=11)
    ax.set_ylabel("Densité", fontsize=11)
    ax.set_xscale(xscale)
    title = "Échelle linéaire (P99 étoiles)" if xscale == "linear" else "Échelle logarithmique (toutes classes)"
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Distribution des erreurs de reconstruction — AE vs classes spectrales",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
err_path = FIGURES_DIR / "ae_error_distribution_logscale.png"
fig.savefig(err_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Sauvegardée : {err_path}")

### Interprétation — L'autoencodeur comme détecteur d'anomalies

La distribution des erreurs révèle une **séparation spectaculaire** entre les classes :

| Classe | n | MSE médiane | Rapport vs étoiles |
|--------|---|------------|--------------------|
| STAR   | 42 862 | 0.281 | ×1 |
| GALAXY | 54 | 1.824 | ×**6.5** |
| QSO    | 4  | 30.06 | ×**107** |

Un simple **seuil sur la MSE de reconstruction** (par exemple MSE > 2.0) permettrait d'isoler les galaxies et QSOs du catalogue stellaire sans aucune supervision ni label. Cette propriété émerge naturellement du fait que l'AE a appris une représentation compressée optimisée pour les étoiles — les autres types d'objets y sont des **outliers** par construction.

> **Application potentielle** : sur le catalogue complet LAMOST DR5 (~9 millions de spectres), ce détecteur pourrait identifier automatiquement les contaminations non-stellaires sans annotation manuelle.

<a id="interp"></a>

<div style="background: linear-gradient(135deg, #3D405B 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(61,64,91,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">5 · Continuité de l'espace latent - interpolation</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(61,64,91,0.10), rgba(90,42,94,0.10));
    border-left: 6px solid #3D405B;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 0 0 10px 0;
">
Une interpolation linéaire entre deux points latents doit produire des reconstructions physiquement cohérentes si la géométrie latente est bien apprise.
</div>

Interpolation entre deux codes latents $z_A$ et $z_B$ :

$$
z(\alpha)=(1-\alpha)z_A + \alpha z_B, \quad \alpha\in[0,1]
$$

Si l’espace latent est régulier, les spectres reconstruits le long de la trajectoire doivent évoluer progressivement (sans sauts non physiques majeurs).

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #3D405B; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>


Une propriété clé d'un bon espace latent est sa **continuité** :
interpoler linéairement entre deux points dans l'espace latent doit
produire des reconstructions physiquement cohérentes.

Ici on interpole entre une **étoile froide** (T_eff ~ 4000 K)
et une **étoile chaude** (T_eff ~ 8000 K) et on observe comment
les features reconstruites évoluent.

In [ ]:
# Sélection d'une étoile froide et d'une étoile chaude
teff = meta["teff_gspphot"].values
valid_teff = np.isfinite(teff)

# Étoile froide : T_eff la plus basse dans le top-SNR
cold_candidates = np.where(valid_teff & (teff < 4200))[0]
hot_candidates  = np.where(valid_teff & (teff > 7500))[0]

if len(cold_candidates) == 0 or len(hot_candidates) == 0:
    print("⚠ Pas assez de candidats — utilisation des percentiles 5 et 95")
    cold_candidates = np.where(valid_teff & (teff < np.nanpercentile(teff, 5)))[0]
    hot_candidates  = np.where(valid_teff & (teff > np.nanpercentile(teff, 95)))[0]

# Prend le meilleur SNR dans chaque groupe
snr_r = meta["snr_r"].values if "snr_r" in meta.columns else np.ones(len(X))

idx_cold = cold_candidates[np.argmax(snr_r[cold_candidates])]
idx_hot  = hot_candidates[np.argmax(snr_r[hot_candidates])]

teff_cold = teff[idx_cold]
teff_hot  = teff[idx_hot]
label_a   = f"Froide (T_eff={teff_cold:.0f} K)"
label_b   = f"Chaude (T_eff={teff_hot:.0f} K)"

print(f"Étoile A (froide) : indice {idx_cold}, T_eff={teff_cold:.0f} K")
print(f"Étoile B (chaude) : indice {idx_hot},  T_eff={teff_hot:.0f} K")

In [ ]:
# ── Interpolation dans l'espace latent ────────────────────────────────────
Z_interp, X_interp = ae.latent_interpolation(
    X[idx_cold], X[idx_hot],
    n_steps=15,
)

fig, axes = viz.plot_latent_interpolation(
    Z_interp, X_interp,
    feature_names=feature_names,
    label_a=label_a,
    label_b=label_b,
    save_path=FIGURES_DIR / "ae_latent_interpolation.png",
)
plt.show()

print("\nTrajectoire latente :")
print(f"  Départ  : z = ({Z_interp[0,0]:.3f}, {Z_interp[0,1]:.3f})")
print(f"  Arrivée : z = ({Z_interp[-1,0]:.3f}, {Z_interp[-1,1]:.3f})")
print(f"  Distance euclidienne dans l'espace latent : "
      f"{np.linalg.norm(Z_interp[-1] - Z_interp[0]):.3f}")

<a id="synthesis"></a>

<div style="background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(90,42,94,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">6 · Synthèse comparative PCA / UMAP / Autoencodeur</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
La figure de synthèse 3x3 et le tableau quantitatif rassemblent les compromis entre interprétabilité, non-linéarité et reconstruction.
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 0 0 10px 0;
">
<strong>6.0 Chargement des embeddings de référence (NB02)</strong>
</div>

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
# ── Chargement des embeddings UMAP/t-SNE (sauvegardés par NB02) ──────────
# NB02 sauvegarde dans phy3500_embeddings.joblib — PAS dans Z_umap.npy.
# Bug corrigé : l'ancien code cherchait un fichier .npy inexistant.

embeddings_path = Path(paths["REPORTS_DIR"]) / "phy3500_embeddings.joblib"

if embeddings_path.exists():
    embeddings_output = joblib.load(embeddings_path)
    Z_umap = embeddings_output["Z_umap"]
    Z_tsne = embeddings_output.get("Z_tsne", None)
    print(f"✓ Embeddings chargés depuis {embeddings_path.name}")
    print(f"  Z_umap : {Z_umap.shape}")
    if Z_tsne is not None:
        print(f"  Z_tsne : {Z_tsne.shape}")
else:
    Z_umap = None
    Z_tsne = None
    print("⚠  Fichier phy3500_embeddings.joblib introuvable.")
    print("   → Lancer d'abord 02_umap_tsne.ipynb (cellule 7 — Sauvegarde).")

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>6.1 Figure de synthèse 3x3 - PCA / UMAP / Autoencodeur</strong><br>
Grille comparative : lignes = méthodes, colonnes = colorations physiques.
</div>

In [ ]:
# ── Figure de synthèse 3×3 : PCA / UMAP / Autoencodeur ──────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

# Palette classes LAMOST (top-level)
CLASS_PALETTE = {
    "STAR":    "#4C72B0",
    "GALAXY":  "#DD8452",
    "QSO":     "#55A868",
    "UNKNOWN": "#8172B3",
}
CLASS_MARKER = {
    "STAR": "o", "GALAXY": "s", "QSO": "^", "UNKNOWN": "x",
}

# Méthodes disponibles (Z_umap peut être None si NB02 non exécuté)
METHODS = []
METHODS.append(("PCA",          scores_pca[:, :2], "PC1",      "PC2"))
if Z_umap is not None:
    METHODS.append(("UMAP",      Z_umap,            "UMAP 1",   "UMAP 2"))
METHODS.append(("Autoencodeur", Z_ae,              "Latent 1", "Latent 2"))

PARAMS = [
    ("Classe spectrale", None,              None),
    ("T_eff (K)",        "teff_gspphot",   "plasma"),
    ("[Fe/H]",           "mh_gspphot",     "RdYlBu_r"),
]

n_rows = len(METHODS)
n_cols = len(PARAMS)
fig = plt.figure(figsize=(5.5 * n_cols, 4.5 * n_rows), dpi=150)
gs  = gridspec.GridSpec(
    n_rows, n_cols, figure=fig,
    hspace=0.45, wspace=0.35,
    left=0.07, right=0.96, top=0.91, bottom=0.06,
)

for row, (method_name, Z_m, xlabel, ylabel) in enumerate(METHODS):
    for col, (param_label, param_col, cmap_name) in enumerate(PARAMS):
        ax = fig.add_subplot(gs[row, col])

        if param_col is None:
            # ── Coloration par classe ─────────────────────────────────────
            for cls in np.unique(y):
                mask = y == cls
                ax.scatter(
                    Z_m[mask, 0], Z_m[mask, 1],
                    c=CLASS_PALETTE.get(cls, "#888888"),
                    marker=CLASS_MARKER.get(cls, "o"),
                    s=2, alpha=0.45, linewidths=0,
                    label=cls, rasterized=True,
                )
            # Légende uniquement sur le premier panneau
            if row == 0:
                ax.legend(
                    markerscale=4, fontsize=8, framealpha=0.8,
                    loc="best", handletextpad=0.3,
                )
        else:
            # ── Coloration par paramètre continu ──────────────────────────
            if param_col in meta.columns:
                vals = meta[param_col].values.astype(float)
                valid = np.isfinite(vals)
                # Points sans paramètre Gaia en gris clair
                if (~valid).any():
                    ax.scatter(
                        Z_m[~valid, 0], Z_m[~valid, 1],
                        c="#dddddd", s=1, alpha=0.15,
                        rasterized=True, linewidths=0,
                    )
                sc = ax.scatter(
                    Z_m[valid, 0], Z_m[valid, 1],
                    c=vals[valid],
                    cmap=cmap_name,
                    vmin=np.nanpercentile(vals[valid], 2),
                    vmax=np.nanpercentile(vals[valid], 98),
                    s=2, alpha=0.55, linewidths=0,
                    rasterized=True,
                )
                plt.colorbar(
                    sc, ax=ax,
                    fraction=0.046, pad=0.04,
                    label=param_label,
                ).ax.tick_params(labelsize=7)

        # ── Cosmétique commune ────────────────────────────────────────────
        ax.set_aspect("equal", "box")
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.2, linewidth=0.5)
        ax.set_xlabel(xlabel, fontsize=8)

        # En-têtes : méthode à gauche, paramètre en haut
        if col == 0:
            ax.set_ylabel(
                f"{method_name}\n{ylabel}",
                fontsize=9, fontweight="bold",
            )
        else:
            ax.set_ylabel(ylabel, fontsize=8)
        if row == 0:
            ax.set_title(param_label, fontsize=11, fontweight="bold", pad=6)

fig.suptitle(
    "Synthèse comparative — PCA / UMAP / Autoencodeur\n"
    "Spectres stellaires LAMOST DR5 × Gaia DR3",
    fontsize=14, fontweight="bold",
)

synth_path = FIGURES_DIR / "synthesis_pca_umap_ae.png"
fig.savefig(synth_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"\n✓ Figure de synthèse sauvegardée : {synth_path}")

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #E5D8EC;
    border-left: 5px solid #5A2A5E;
    border-radius: 8px;
    padding: 10px 12px;
    margin: 8px 0;
">
<strong>6.2 Tableau comparatif quantitatif</strong><br>
Corrélations de température, erreur de reconstruction et propriétés qualitatives par méthode.
</div>

In [ ]:
# ── Tableau comparatif quantitatif ──────────────────────────────────────
from scipy.stats import spearmanr

teff_vals  = meta["teff_gspphot"].values.astype(float)
teff_valid = np.isfinite(teff_vals)

def _spearman_ax(Z_m, axis=0):
    """Corrélation Spearman entre l'axe `axis` de Z_m et T_eff."""
    if Z_m is None:
        return float('nan')
    r, _ = spearmanr(Z_m[teff_valid, axis], teff_vals[teff_valid])
    return r

results = {}

# PCA
mse_pca2 = float(pca.reconstruction_error(X, n_components=2).mean())
r_pca = _spearman_ax(scores_pca)
results["PCA (PC1/PC2)"] = {
    "ρ(axe 1, T_eff)": f"{r_pca:+.3f}",
    "MSE recon. (2 dims)": f"{mse_pca2:.5f}",
    "Paramétrique": "Oui",
    "Non-linéaire": "Non",
    "Interprétable": "Oui (loadings)",
}

# Autoencodeur
mse_ae = float(ae.reconstruction_mse(X))
r_ae   = _spearman_ax(Z_ae)
results["Autoencodeur (z=2)"] = {
    "ρ(axe 1, T_eff)": f"{r_ae:+.3f}",
    "MSE recon. (2 dims)": f"{mse_ae:.5f}",
    "Paramétrique": "Oui",
    "Non-linéaire": "Oui",
    "Interprétable": "Partielle",
}

# UMAP
if Z_umap is not None:
    r_umap = _spearman_ax(Z_umap)
    results["UMAP"] = {
        "ρ(axe 1, T_eff)": f"{r_umap:+.3f}",
        "MSE recon. (2 dims)": "N/A",
        "Paramétrique": "Oui (partiel)",
        "Non-linéaire": "Oui",
        "Interprétable": "Non",
    }

df_summary = pd.DataFrame(results).T
print("\n" + "=" * 65)
print("  RÉSUMÉ — NB03 : Comparaison PCA / UMAP / Autoencodeur")
print("=" * 65)
print(df_summary.to_string())
print("=" * 65)
print(f"\n  Temps entraînement AE : {ae.fit_time_:.1f} s")
print(f"  Epochs effectuées     : {len(ae.history_['train_loss'])}")
print(f"  Best val_loss         : {min(ae.history_['val_loss']):.6f}")

<a id="save"></a>

<div style="background: linear-gradient(135deg, #2C6E8F 0%, #5A2A5E 100%); padding: 16px 22px; border-radius: 10px; margin: 16px 0 10px 0; box-shadow: 0 6px 18px rgba(44,110,143,0.25);">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 1px;">7 · Sauvegarde des sorties et rapports de run</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(44,110,143,0.10), rgba(90,42,94,0.10));
    border-left: 6px solid #2C6E8F;
    border-radius: 10px;
    padding: 14px 16px;
    margin: 0 0 10px 0;
">
Cette section enregistre automatiquement les artefacts du run (joblib, JSON, TXT), avec un fichier horodaté et une version latest.
</div>

### Pourquoi c’est important
- Traçabilité complète des hyperparamètres, métriques et figures.
- Réutilisation directe des artefacts dans les notebooks suivants et le rapport.
- Reproductibilité stricte des résultats expérimentaux.

<div style="text-align: right; margin: 0 0 10px 0;">
  <a href="#toc" style="color: #2C6E8F; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

In [ ]:
import json
import joblib
from datetime import datetime, timezone


def _class_counts(labels):
    classes, counts = np.unique(labels, return_counts=True)
    return {str(cls): int(cnt) for cls, cnt in zip(classes, counts)}


def _safe_df_records(df, max_rows=None):
    if df is None:
        return None
    out = df.copy()
    if max_rows is not None:
        out = out.head(max_rows)
    out = out.where(pd.notna(out), None)
    return out.to_dict(orient="records")


def _json_default(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (datetime, pd.Timestamp)):
        return obj.isoformat()
    raise TypeError(f"Type non serialisable: {type(obj)!r}")


def _embedding_stats(Z):
    if Z is None:
        return None
    arr = np.asarray(Z)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return {"shape": list(arr.shape)}
    return {
        "shape": list(arr.shape),
        "x_min": float(np.nanmin(arr[:, 0])),
        "x_max": float(np.nanmax(arr[:, 0])),
        "y_min": float(np.nanmin(arr[:, 1])),
        "y_max": float(np.nanmax(arr[:, 1])),
        "x_mean": float(np.nanmean(arr[:, 0])),
        "y_mean": float(np.nanmean(arr[:, 1])),
        "x_std": float(np.nanstd(arr[:, 0])),
        "y_std": float(np.nanstd(arr[:, 1])),
    }


def _fmt_seconds(value):
    if value is None:
        return "N/A"
    try:
        v = float(value)
    except (TypeError, ValueError):
        return "N/A"
    return f"{v:.2f}s" if np.isfinite(v) else "N/A"


required_vars = ["X", "y", "meta", "ae", "feature_names", "paths", "FIGURES_DIR"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        "Variables manquantes pour la sauvegarde. "
        "Executer les cellules precedentes du notebook avant cette cellule.\n"
        f"Manquantes: {missing}"
    )


timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = Path(paths["REPORTS_DIR"]) / "runs" / "phy3500_autoencoder"
run_dir.mkdir(parents=True, exist_ok=True)

Z_ae_current = Z_ae if "Z_ae" in globals() else ae.encode(X)
X_recon_current = X_recon if "X_recon" in globals() else None
history_current = history if "history" in globals() else getattr(ae, "history_", None)

train_loss = history_current.get("train_loss", []) if isinstance(history_current, dict) else []
val_loss = history_current.get("val_loss", []) if isinstance(history_current, dict) else []

ae_fit_time = getattr(ae, "fit_time_", None)
epochs_done = len(train_loss) if isinstance(train_loss, list) else None
best_val_loss = float(min(val_loss)) if len(val_loss) > 0 else None
final_train_loss = float(train_loss[-1]) if len(train_loss) > 0 else None
final_val_loss = float(val_loss[-1]) if len(val_loss) > 0 else None

try:
    mse_ae_global = float(ae.reconstruction_mse(X))
except Exception:
    mse_ae_global = None

n_pcs_95_current = int(n_pcs_95) if "n_pcs_95" in globals() else None
mse_pca2 = None
mse_pca95 = None
if "pca" in globals():
    try:
        mse_pca2 = float(pca.reconstruction_error(X, n_components=2).mean())
        if n_pcs_95_current is None:
            n_pcs_95_current = int(pca.n_components_for_variance(0.95))
        mse_pca95 = float(pca.reconstruction_error(X, n_components=n_pcs_95_current).mean())
    except Exception:
        pass

phys_cols = ["teff_gspphot", "logg_gspphot", "mh_gspphot", "bp_rp"]
ae_correlations = {}
for col in phys_cols:
    if col not in meta.columns:
        continue
    vals = pd.to_numeric(meta[col], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(vals)
    if valid.sum() < 3:
        ae_correlations[col] = {"latent_axis_1": None, "latent_axis_2": None}
        continue
    r1 = pd.Series(Z_ae_current[valid, 0]).corr(pd.Series(vals[valid]), method="spearman")
    r2 = pd.Series(Z_ae_current[valid, 1]).corr(pd.Series(vals[valid]), method="spearman")
    ae_correlations[col] = {
        "latent_axis_1": float(r1) if pd.notna(r1) else None,
        "latent_axis_2": float(r2) if pd.notna(r2) else None,
    }

latent_distance = None
if "Z_interp" in globals():
    zi = np.asarray(Z_interp)
    if zi.ndim == 2 and zi.shape[0] >= 2:
        latent_distance = float(np.linalg.norm(zi[-1] - zi[0]))

autoencoder_output = {
    "Z_ae": Z_ae_current,
    "X_recon": X_recon_current,
    "y": y,
    "meta": meta,
    "feature_names": feature_names,
    "history": history_current,
    "comparison_df": comparison_df if "comparison_df" in globals() else None,
    "reconstruction_summary": summary if "summary" in globals() else None,
    "method_summary": df_summary if "df_summary" in globals() else None,
    "scores_pca": scores_pca if "scores_pca" in globals() else None,
    "Z_umap": Z_umap if "Z_umap" in globals() else None,
    "Z_tsne": Z_tsne if "Z_tsne" in globals() else None,
    "Z_interp": Z_interp if "Z_interp" in globals() else None,
    "X_interp": X_interp if "X_interp" in globals() else None,
    "idx_cold": int(idx_cold) if "idx_cold" in globals() else None,
    "idx_hot": int(idx_hot) if "idx_hot" in globals() else None,
    "teff_cold": float(teff_cold) if "teff_cold" in globals() else None,
    "teff_hot": float(teff_hot) if "teff_hot" in globals() else None,
    "n_pcs_95": n_pcs_95_current,
    "ae_fit_time_s": float(ae_fit_time) if ae_fit_time is not None else None,
    "ae_mse_global": mse_ae_global,
}

save_path = Path(paths["REPORTS_DIR"]) / "phy3500_autoencoder_output.joblib"
joblib.dump(autoencoder_output, save_path)

report = {
    "run_timestamp_utc": timestamp,
    "paths": {
        "features_path": str(FEATURES_PATH) if "FEATURES_PATH" in globals() else None,
        "catalog_path": str(CATALOG_PATH) if "CATALOG_PATH" in globals() else None,
        "figures_dir": str(FIGURES_DIR),
        "models_dir": str(MODELS_DIR) if "MODELS_DIR" in globals() else None,
        "ae_model_path": str(ae_model_path) if "ae_model_path" in globals() else None,
        "autoencoder_joblib_path": str(save_path),
        "run_dir": str(run_dir),
    },
    "shapes": {
        "X": list(np.asarray(X).shape),
        "Z_ae": list(np.asarray(Z_ae_current).shape),
        "X_recon": list(np.asarray(X_recon_current).shape) if X_recon_current is not None else None,
        "scores_pca": list(np.asarray(scores_pca).shape) if "scores_pca" in globals() else None,
        "meta": list(meta.shape),
    },
    "class_counts": _class_counts(y),
    "metrics": {
        "ae_fit_time_s": float(ae_fit_time) if ae_fit_time is not None else None,
        "ae_epochs_done": int(epochs_done) if epochs_done is not None else None,
        "ae_best_val_loss": best_val_loss,
        "ae_final_train_loss": final_train_loss,
        "ae_final_val_loss": final_val_loss,
        "ae_mse_global": mse_ae_global,
        "pca_mse_2_components": mse_pca2,
        "pca_mse_n_components_95": mse_pca95,
        "n_pcs_95": n_pcs_95_current,
    },
    "correlations": {
        "ae_latent_vs_physical_spearman": ae_correlations,
        "pca_corr_table": _safe_df_records(corr_pca.reset_index()) if "corr_pca" in globals() else None,
    },
    "tables": {
        "comparison_df": _safe_df_records(comparison_df) if "comparison_df" in globals() else None,
        "reconstruction_summary": _safe_df_records(summary) if "summary" in globals() else None,
        "method_summary": (
            _safe_df_records(df_summary.reset_index().rename(columns={"index": "method"}))
            if "df_summary" in globals()
            else None
        ),
    },
    "interpolation": {
        "idx_cold": int(idx_cold) if "idx_cold" in globals() else None,
        "idx_hot": int(idx_hot) if "idx_hot" in globals() else None,
        "teff_cold": float(teff_cold) if "teff_cold" in globals() else None,
        "teff_hot": float(teff_hot) if "teff_hot" in globals() else None,
        "latent_distance": latent_distance,
    },
    "embeddings": {
        "umap_available": bool("Z_umap" in globals() and Z_umap is not None),
        "tsne_available": bool("Z_tsne" in globals() and Z_tsne is not None),
        "umap_stats": _embedding_stats(Z_umap) if "Z_umap" in globals() else None,
        "tsne_stats": _embedding_stats(Z_tsne) if "Z_tsne" in globals() else None,
        "ae_stats": _embedding_stats(Z_ae_current),
    },
    "figures_saved": sorted(str(p) for p in FIGURES_DIR.glob("*.png")),
}

json_path = run_dir / f"phy3500_autoencoder_run_{timestamp}.json"
json_latest_path = run_dir / "phy3500_autoencoder_run_latest.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=_json_default)

with open(json_latest_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=_json_default)

text_lines = [
    "AstroSpectro | PHY-3500 Autoencoder Run Report",
    "=" * 72,
    f"Timestamp (UTC)             : {timestamp}",
    f"Features file               : {FEATURES_PATH if 'FEATURES_PATH' in globals() else 'N/A'}",
    f"Catalog file                : {CATALOG_PATH if 'CATALOG_PATH' in globals() else 'N/A'}",
    f"Figures directory           : {FIGURES_DIR}",
    "",
    "[Saved artifacts]",
    f"- Autoencoder joblib        : {save_path}",
    f"- JSON run report           : {json_path}",
    "",
    "[Shapes]",
    f"- X                         : {np.asarray(X).shape}",
    f"- Z_ae                      : {np.asarray(Z_ae_current).shape}",
    f"- X_recon                   : {np.asarray(X_recon_current).shape if X_recon_current is not None else 'N/A'}",
    "",
    "[Class counts]",
    f"- y                         : {_class_counts(y)}",
    "",
    "[Metrics]",
    f"- AE fit time               : {_fmt_seconds(ae_fit_time)}",
    f"- AE epochs                 : {epochs_done if epochs_done is not None else 'N/A'}",
    f"- AE best val_loss          : {best_val_loss if best_val_loss is not None else 'N/A'}",
    f"- AE final train_loss       : {final_train_loss if final_train_loss is not None else 'N/A'}",
    f"- AE final val_loss         : {final_val_loss if final_val_loss is not None else 'N/A'}",
    f"- AE MSE global             : {mse_ae_global if mse_ae_global is not None else 'N/A'}",
    f"- PCA MSE (2)               : {mse_pca2 if mse_pca2 is not None else 'N/A'}",
    f"- PCA MSE (n95)             : {mse_pca95 if mse_pca95 is not None else 'N/A'}",
    f"- n_pcs_95                  : {n_pcs_95_current if n_pcs_95_current is not None else 'N/A'}",
    "",
    "[AE latent correlations - Spearman]",
    str(ae_correlations),
    "",
]

if "comparison_df" in globals():
    text_lines.extend([
        "Comparison AE vs PCA:",
        comparison_df.round(6).to_string(index=False),
        "",
    ])

if "summary" in globals():
    text_lines.extend([
        "Reconstruction summary by class:",
        summary.round(6).to_string(index=False),
        "",
    ])

if "df_summary" in globals():
    text_lines.extend([
        "Method summary table:",
        df_summary.to_string(),
        "",
    ])

if latent_distance is not None:
    text_lines.extend([
        "Interpolation:",
        f"- idx_cold                  : {int(idx_cold) if 'idx_cold' in globals() else 'N/A'}",
        f"- idx_hot                   : {int(idx_hot) if 'idx_hot' in globals() else 'N/A'}",
        f"- latent_distance           : {latent_distance:.6f}",
        "",
    ])

text_path = run_dir / f"phy3500_autoencoder_run_{timestamp}.txt"
text_latest_path = run_dir / "phy3500_autoencoder_run_latest.txt"
text_content = "\n".join(text_lines)

with open(text_path, "w", encoding="utf-8") as f:
    f.write(text_content)

with open(text_latest_path, "w", encoding="utf-8") as f:
    f.write(text_content)

print(f"Autoencoder output sauvegarde -> {save_path}")
print(f"Rapport JSON run              -> {json_path}")
print(f"Rapport TXT run               -> {text_path}")
print(f"Raccourci JSON latest         -> {json_latest_path}")
print(f"Raccourci TXT latest          -> {text_latest_path}")
print(f"  Z_ae shape                  : {np.asarray(Z_ae_current).shape}")

<a id="conclusion"></a>

<div style="
    background: linear-gradient(135deg, #2D1E2F 0%, #5A2A5E 50%, #2C6E8F 100%);
    padding: 18px 22px;
    border-radius: 12px;
    margin: 18px 0 12px 0;
    box-shadow: 0 8px 24px rgba(45,30,47,0.30);
">
  <h2 style="color: white; margin: 0; font-weight: 350; letter-spacing: 1px;">Résumé final (run 20260402T171003Z)</h2>
</div>

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 10px;
    padding: 12px 14px;
    margin-bottom: 12px;
">
Tableau complété à partir de <code>data/reports/runs/phy3500_autoencoder/phy3500_autoencoder_run_20260402T171003Z.json</code>.
</div>

| Analyse | Résultat clé (run 20260402T171003Z) |
|---------|--------------------------------------|
| Échantillon | 42 920 objets, 208 features standardisées |
| Entraînement AE | 190.86 s, 190 epochs, best val_loss = 0.51835 |
| Reconstruction AE (z=2) | MSE globale = 0.54111 |
| Référence PCA (2 composantes) | MSE = 0.73167 |
| Référence PCA (n=51, 95% var) | MSE = 0.24428 |
| Corrélation AE axe 2 - T_eff | rho = -0.756 (gradient thermique fort) |
| Corrélation AE axe 2 - BP-RP | rho = +0.782 (cohérence couleur-température) |
| Interpolation latente froide-chaude | distance = 12.841 ; 4199.6 K -> 7984.0 K |

### Lecture scientifique
- À 2 dimensions, l’autoencodeur non linéaire reconstruit mieux que PCA-2 (0.541 vs 0.732), ce qui confirme l’intérêt d’un modèle non linéaire sous forte contrainte dimensionnelle.
- PCA avec davantage de composantes reste naturellement meilleur en erreur pure (MSE 0.244 à 51 composantes), ce qui est attendu car la capacité de reconstruction augmente avec la dimension.
- Les corrélations latentes montrent que l’information physique est captée mais redistribuée sur des axes non directement interprétables comme PC1/PC2.
- Les classes minoritaires (GALAXY, QSO) présentent des erreurs bien plus élevées que STAR, en partie à cause du déséquilibre de classes.

### Recommandation pipeline
Pour la modélisation supervisée principale, les features physiques (ou PCA interprétable) restent la base la plus robuste. L’autoencodeur est particulièrement pertinent pour l’exploration non supervisée, la continuité latente et la détection d’objets atypiques.

<div style="text-align: right; margin-top: 10px;">
  <a href="#toc" style="color: #5A2A5E; text-decoration: none; font-size: 12px;">Retour à la table des matières</a>
</div>

<div style="
    background: linear-gradient(135deg, #5A2A5E 0%, #2C6E8F 100%);
    padding: 16px 20px;
    border-radius: 10px;
    margin: 18px 0 10px 0;
    box-shadow: 0 6px 18px rgba(90,42,94,0.25);
">
  <h2 style="color: white; margin: 0; font-weight: 400; letter-spacing: 0.8px;">Références scientifiques et techniques</h2>
</div>

### Fondements autoencodeur et apprentissage profond
1. Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). *Learning representations by back-propagating errors*. Nature, 323, 533-536. (article fondateur de la rétropropagation).
2. Hinton, G. E., & Salakhutdinov, R. R. (2006). *Reducing the dimensionality of data with neural networks*. Science, 313(5786), 504-507. (autoencodeurs pour réduction de dimension).
3. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. (théorie générale des réseaux profonds, régularisation, optimisation).

### Réduction de dimension de référence
4. Jolliffe, I. T., & Cadima, J. (2016). *Principal component analysis: a review and recent developments*. Phil. Trans. A, 374:20150202. (cadre théorique PCA).
5. McInnes, L., Healy, J., & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*. arXiv:1802.03426. (comparaison conceptuelle avec embeddings non linéaires).

### Corrélation de rang et interprétation physique
6. Spearman, C. (1904). *The proof and measurement of association between two things*. American Journal of Psychology, 15(1), 72-101. (définition de rho de Spearman).

### Contexte astrophysique des données
7. Gaia Collaboration (2023). *Gaia Data Release 3: Summary of the content and survey properties*. Astronomy & Astrophysics, 674, A1.
8. Luo, A.-L., et al. (2019). *The LAMOST DR5 release*. Research in Astronomy and Astrophysics.

### Implémentation Python
9. PyTorch developers. *PyTorch documentation*. https://pytorch.org/docs/stable/
10. scikit-learn developers. *PCA documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

<div style="
    background: linear-gradient(135deg, rgba(52, 94, 133, 0.18), rgba(89, 122, 157, 0.14));
    border: 1px solid #D4E3EC;
    border-radius: 8px;
    padding: 10px 12px;
    margin-top: 12px;
">
Ces références couvrent les briques mobilisées dans ce notebook : apprentissage latent non linéaire, comparaison avec méthodes de réduction de dimension, validation statistique et contexte astrophysique.
</div>

## Conclusions

### Ce que chaque méthode apporte

| Critère | PCA | UMAP | Autoencodeur |
|---------|-----|------|--------------|
| Linéarité | Linéaire | Non-linéaire | Non-linéaire |
| Paramétrique (transform nouveaux points) | ✓ | ✓ (partiel) | ✓ |
| Erreur de reconstruction (2 dims) | *voir tableau* | N/A | *voir tableau* |
| Interprétabilité des axes | ✓✓ (loadings physiques) | ✗ | ✗ |
| Temps d'exécution | < 1 s | *voir §2* | *voir §2* |
| Stabilité | ✓✓ (déterministe) | *voir stabilité* | Variable |

### Observations physiques

- **PC1** corrèle fortement avec T_eff (température) : la PCA retrouve
  la séquence principale de Hertzsprung-Russell sans supervision.
- **UMAP** révèle des sous-structures discrètes (clusters) invisibles
  en PCA, témoignant de non-linéarités dans l'espace spectral.
- **L'autoencodeur** compresse vers un espace latent continu (validé
  par l'interpolation §5) mais ses axes n'ont pas d'interprétation
  physique directe contrairement aux loadings de la PCA.

### Recommandation pour le pipeline AstroSpectro principal

Pour la **classification XGBoost**, les features physiques directes (V2)
restent préférables : plus interprétables, déterministes, et prouvées
efficaces à 84 %+. La réduction de dimension est plus pertinente pour
l'**exploration non supervisée** et la **détection d'anomalies**.

### Lien avec PHY-3500

Ce notebook illustre le spectre des méthodes numériques enseignées :
- PCA → décomposition spectrale (valeurs propres, SVD, NR chap. 11)
- UMAP → graphes de voisinage, optimisation stochastique
- Autoencodeur → rétropropagation, descente de gradient, régularisation